### **LANGKAH 1: Inisialisasi Environment, Google Drive, & Clone Repositori LASFNet**

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Folder target di Google Drive siap: {SAVE_DIR}")

%cd /content
!rm -rf /content/LASFNet
!git clone https://github.com/leileilei2000/LASFNet.git
%cd /content/LASFNet

!pip install -q "scikit-image>=0.24.0" xlsxwriter huggingface_hub thop pycocotools ultralytics albumentations timm einops prettytable
print("Instalasi dependensi selesai!")

Mounted at /content/drive
Folder target di Google Drive siap: /content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run
/content
Cloning into 'LASFNet'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (260/260), done.
remote: Total 269 (delta 88), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 5.88 MiB | 15.56 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/LASFNet
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.5/179.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 12.8 MB/s eta 0:00:00
Instalasi dependensi selesai!


### **LANGKAH 2: Unduh Dataset McCheng/DroneVehicle dari Hugging Face**

In [4]:
import os
import subprocess
from huggingface_hub import snapshot_download

target_dir = "/content/LASFNet/dataset/dronevehicle_new"
os.makedirs(target_dir, exist_ok=True)

print("Mengunduh dataset dari Hugging Face...")
snapshot_download(repo_id="McCheng/DroneVehicle", repo_type="dataset", local_dir=target_dir)

for zip_name in ['train.zip', 'val.zip', 'test.zip']:
    zip_path = os.path.join(target_dir, zip_name)
    if os.path.exists(zip_path):
        print(f"Mengekstrak {zip_name}...")
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", target_dir])

print("Ekstraksi dataset selesai!")

Mengunduh dataset dari Hugging Face...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Mengekstrak train.zip...
Mengekstrak val.zip...
Mengekstrak test.zip...
Ekstraksi dataset selesai!


### **LANGKAH 3: Penataan Struktur Direktori & Pembersihan File Bermasalah Secara Manual**

In [5]:
import os
import shutil

target_dir = "/content/LASFNet/dataset/dronevehicle_new"

for mode in ['train', 'val', 'test']:
    os.makedirs(f"{target_dir}/data_rgb/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/labels", exist_ok=True)
    os.makedirs(f"{target_dir}/data_rgb/{mode}/images", exist_ok=True)
    os.makedirs(f"{target_dir}/data_ir/{mode}/images", exist_ok=True)

print("Menata letak file gambar dan label...")
for mode in ['train', 'val', 'test']:
    !mv {target_dir}/{mode}/{mode}img/* {target_dir}/data_rgb/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}imgr/* {target_dir}/data_ir/{mode}/images/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}label/* {target_dir}/data_rgb/{mode}/labels/ 2>/dev/null || true
    !mv {target_dir}/{mode}/{mode}labelr/* {target_dir}/data_ir/{mode}/labels/ 2>/dev/null || true

problematic_files = ["04895", "06463", "14048", "12900", "05001", "09012", "04991", "16009"]

print("\n--- Menghapus File Bermasalah secara Manual ---")
for base_name in problematic_files:
    for mode in ['train', 'val', 'test']:
        paths_to_remove = [
            f"data_rgb/{mode}/images/{base_name}.jpg",
            f"data_ir/{mode}/images/{base_name}.jpg",
            f"data_rgb/{mode}/labels/{base_name}.txt",
            f"data_ir/{mode}/labels/{base_name}.txt",
            f"data_rgb/{mode}/labels/{base_name}.xml",
            f"data_ir/{mode}/labels/{base_name}.xml"
        ]
        for p in paths_to_remove:
            full_path = os.path.join(target_dir, p)
            if os.path.exists(full_path):
                os.remove(full_path)
                print(f"Dihapus: {p}")

print("Pembersihan file manual selesai!")

Menata letak file gambar dan label...

--- Menghapus File Bermasalah secara Manual ---
Dihapus: data_rgb/train/images/04895.jpg
Dihapus: data_ir/train/images/04895.jpg
Dihapus: data_rgb/train/labels/04895.xml
Dihapus: data_ir/train/labels/04895.xml
Dihapus: data_rgb/test/images/04895.jpg
Dihapus: data_ir/test/images/04895.jpg
Dihapus: data_rgb/test/labels/04895.xml
Dihapus: data_ir/test/labels/04895.xml
Dihapus: data_rgb/train/images/06463.jpg
Dihapus: data_ir/train/images/06463.jpg
Dihapus: data_rgb/train/labels/06463.xml
Dihapus: data_ir/train/labels/06463.xml
Dihapus: data_rgb/test/images/06463.jpg
Dihapus: data_ir/test/images/06463.jpg
Dihapus: data_rgb/test/labels/06463.xml
Dihapus: data_ir/test/labels/06463.xml
Dihapus: data_rgb/train/images/14048.jpg
Dihapus: data_ir/train/images/14048.jpg
Dihapus: data_rgb/train/labels/14048.xml
Dihapus: data_ir/train/labels/14048.xml
Dihapus: data_rgb/train/images/12900.jpg
Dihapus: data_ir/train/images/12900.jpg
Dihapus: data_rgb/train/labels

### **LANGKAH 4: Konversi Format XML DroneVehicle ke YOLO TXT & Sinkronisasi Multimodal**

In [6]:
import os
import xml.etree.ElementTree as ET
import glob

def convert_dronevehicle_xml_to_yolo(xml_dir, txt_dir, width=840, height=712):
    if not os.path.exists(xml_dir):
        return
    os.makedirs(txt_dir, exist_ok=True)
    class_map = {
        'car': 0,
        'truck': 1,
        'bus': 2,
        'van': 3,
        'freight car': 4,
        'freight_car': 4,
        'feright car': 4,
        'feright_car': 4,
        'feright': 4
    }
    xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))

    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        filename = os.path.splitext(os.path.basename(xml_file))[0]
        txt_file_path = os.path.join(txt_dir, f"{filename}.txt")

        yolo_lines = []
        for obj in root.findall('object'):
            name = obj.find('name').text.strip().lower()
            if name not in class_map:
                continue
            class_id = class_map[name]
            polygon = obj.find('polygon')
            bndbox = obj.find('bndbox')
            if polygon is not None:
                try:
                    xs = [float(polygon.find(f'x{i}').text) for i in range(1, 5)]
                    ys = [float(polygon.find(f'y{i}').text) for i in range(1, 5)]
                    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
                    x_center = ((xmin + xmax) / 2.0) / width
                    y_center = ((ymin + ymax) / 2.0) / height
                    box_w = (xmax - xmin) / width
                    box_h = (ymax - ymin) / height
                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                except:
                    continue
            elif bndbox is not None:
                try:
                    xmin = float(bndbox.find('xmin').text)
                    ymin = float(bndbox.find('ymin').text)
                    xmax = float(bndbox.find('xmax').text)
                    ymax = float(bndbox.find('ymax').text)
                    x_center = ((xmin + xmax) / 2.0) / width
                    y_center = ((ymin + ymax) / 2.0) / height
                    box_w = (xmax - xmin) / width
                    box_h = (ymax - ymin) / height
                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                except:
                    continue
        with open(txt_file_path, "w") as f:
            f.write("\n".join(yolo_lines))

target_dir = "/content/LASFNet/dataset/dronevehicle_new"
print("Mengonversi XML ke format TXT YOLO...")
for mode in ['train', 'val', 'test']:
    convert_dronevehicle_xml_to_yolo(f"{target_dir}/data_rgb/{mode}/labels", f"{target_dir}/data_rgb/{mode}/labels")
    convert_dronevehicle_xml_to_yolo(f"{target_dir}/data_ir/{mode}/labels", f"{target_dir}/data_ir/{mode}/labels")
    !find {target_dir}/data_rgb/{mode}/labels -name "*.xml" -delete 2>/dev/null || true
    !find {target_dir}/data_ir/{mode}/labels -name "*.xml" -delete 2>/dev/null || true

def synchronize_dataset(base_dir):
    for mode in ['train', 'val', 'test']:
        print(f"\n--- Sinkronisasi Dataset: {mode.upper()} ---")
        rgb_img_dir = os.path.join(base_dir, f"data_rgb/{mode}/images")
        rgb_lbl_dir = os.path.join(base_dir, f"data_rgb/{mode}/labels")
        ir_img_dir = os.path.join(base_dir, f"data_ir/{mode}/images")
        ir_lbl_dir = os.path.join(base_dir, f"data_ir/{mode}/labels")

        rgb_imgs = {os.path.splitext(f)[0] for f in os.listdir(rgb_img_dir) if f.lower().endswith(('.jpg', '.jpeg'))} if os.path.exists(rgb_img_dir) else set()
        rgb_lbls = {os.path.splitext(f)[0] for f in os.listdir(rgb_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(rgb_lbl_dir) else set()
        ir_imgs = {os.path.splitext(f)[0] for f in os.listdir(ir_img_dir) if f.lower().endswith(('.jpg', '.jpeg'))} if os.path.exists(ir_img_dir) else set()
        ir_lbls = {os.path.splitext(f)[0] for f in os.listdir(ir_lbl_dir) if f.lower().endswith('.txt')} if os.path.exists(ir_lbl_dir) else set()

        valid_keys = rgb_imgs.intersection(rgb_lbls).intersection(ir_imgs).intersection(ir_lbls)
        print(f"Jumlah sampel sinkron sempurna: {len(valid_keys)}")

        def clean_unpaired(directory, valid_set, exts):
            if not os.path.exists(directory): return
            for f in os.listdir(directory):
                name, ext = os.path.splitext(f)
                if ext.lower() in exts and name not in valid_set:
                    try: os.remove(os.path.join(directory, f))
                    except: pass

        clean_unpaired(rgb_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(rgb_lbl_dir, valid_keys, ['.txt'])
        clean_unpaired(ir_img_dir, valid_keys, ['.jpg', '.jpeg', '.png'])
        clean_unpaired(ir_lbl_dir, valid_keys, ['.txt'])

synchronize_dataset(target_dir)
# Bersihkan sisa cache agar scanning ulang aman
!find {target_dir} -name "*.cache" -delete 2>/dev/null || true
print("Dataset terverifikasi, sinkron, dan bersih!")

Mengonversi XML ke format TXT YOLO...

--- Sinkronisasi Dataset: TRAIN ---
Jumlah sampel sinkron sempurna: 17954

--- Sinkronisasi Dataset: VAL ---
Jumlah sampel sinkron sempurna: 1469

--- Sinkronisasi Dataset: TEST ---
Jumlah sampel sinkron sempurna: 8976
Dataset terverifikasi, sinkron, dan bersih!


### **LANGKAH 5: Injeksi Patch Keamanan Code (Autoupdate Bypass & PyTorch Load Patch)**

In [7]:
import os

def inject_bypass(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            code = f.read()
        if "def check_requirements" not in code:
            bypass_code = "import sys\nimport utils.general as ug\num.check_requirements = lambda *args, **kwargs: True\n"
            with open(file_path, "w") as f:
                f.write(bypass_code + "\n" + code)
            print(f"Bypass autoupdate terpasang di {file_path}!")

inject_bypass("/content/LASFNet/train.py")
inject_bypass("/content/LASFNet/val.py")

train_py = "/content/LASFNet/train.py"
with open(train_py, "r") as f:
    content = f.read()
patch_code = """import torch
_orig_load = torch.load
torch.load = lambda *args, **kwargs: _orig_load(*args, **{**kwargs, 'weights_only': False})
"""
if "_orig_load = torch.load" not in content:
    with open(train_py, "w") as f:
        f.write(patch_code + "\n" + content)
    print("Patch torch.load berhasil diterapkan ke train.py!")

Bypass autoupdate terpasang di /content/LASFNet/train.py!
Bypass autoupdate terpasang di /content/LASFNet/val.py!
Patch torch.load berhasil diterapkan ke train.py!


In [8]:
import os

def fix_autoupdate_bypass(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            code = f.read()

        if "um.check_requirements" in code:
            code = code.replace(
                "import sys\nimport utils.general as ug\num.check_requirements = lambda *args, **kwargs: True\n",
                ""
            )
            code = code.replace("um.check_requirements = lambda *args, **kwargs: True", "")

        correct_bypass = """import sys
import utils.general as ug
ug.check_requirements = lambda *args, **kwargs: True
"""
        if "ug.check_requirements = lambda" not in code:
            code = correct_bypass + "\n" + code

        with open(file_path, "w") as f:
            f.write(code)
        print(f"Bypass autoupdate berhasil dikoreksi dan dipasang dengan benar pada: {file_path}!")
    else:
        print(f"File tidak ditemukan: {file_path}")

fix_autoupdate_bypass("/content/LASFNet/train.py")
fix_autoupdate_bypass("/content/LASFNet/val.py")

Bypass autoupdate berhasil dikoreksi dan dipasang dengan benar pada: /content/LASFNet/train.py!
Bypass autoupdate berhasil dikoreksi dan dipasang dengan benar pada: /content/LASFNet/val.py!


In [9]:
import os

def apply_robust_torch_patch(file_path):
    if not os.path.exists(file_path):
        print(f"File tidak ditemukan: {file_path}")
        return

    with open(file_path, "r") as f:
        content = f.read()

    # Hapus patch lama jika ada agar tidak tumpang tindih
    old_patches = [
        "import torch\n_orig_load = torch.load\ntorch.load = lambda *args, **kwargs: _orig_load(*args, **{**kwargs, 'weights_only': False})",
        "_orig_load = torch.load",
        "torch.load = lambda *args, **kwargs:"
    ]
    for op in old_patches:
        if op in content:
            content = content.replace(op, "")

    # Patch baru yang sangat aman untuk PyTorch versi baru
    robust_patch = """import torch
import numpy as np
try:
    # Izinkan rekonstruksi array numpy secara aman di PyTorch 2.6+
    import torch.serialization
    torch.serialization.add_safe_globals([np._core.multiarray._reconstruct, np.ndarray, np.dtype])
except Exception:
    pass

_orig_load = torch.load
def _patched_load(*args, **kwargs):
    if 'weights_only' in kwargs:
        kwargs['weights_only'] = False
    else:
        kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load
"""

    if "_patched_load" not in content:
        with open(file_path, "w") as f:
            f.write(robust_patch + "\n" + content)
        print(f"Patch keamanan PyTorch berhasil diterapkan di: {file_path}!")
    else:
        print(f"Patch sudah terpasang di: {file_path}")

apply_robust_torch_patch("/content/LASFNet/train.py")
apply_robust_torch_patch("/content/LASFNet/val.py")

Patch keamanan PyTorch berhasil diterapkan di: /content/LASFNet/train.py!
Patch keamanan PyTorch berhasil diterapkan di: /content/LASFNet/val.py!


In [10]:
!pip install -q gitpython
print("Pustaka 'gitpython' berhasil dipasang! Silakan jalankan kembali sel training.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 8.1 MB/s eta 0:00:00
Pustaka 'gitpython' berhasil dipasang! Silakan jalankan kembali sel training.


### **LANGKAH 6: Konfigurasi File YAML Dataset**

In [11]:
import os

yaml_content = """# Dataset DroneVehicle untuk LASFNet
path: /content/LASFNet/dataset/dronevehicle_new
train_rgb: data_rgb/train/images
train_ir: data_ir/train/images
val_rgb: data_rgb/val/images
val_ir: data_ir/val/images
test_rgb: data_rgb/test/images
test_ir: data_ir/test/images

# Classes
nc: 5
names:
  0: car
  1: truck
  2: bus
  3: van
  4: freight car
"""

os.makedirs("/content/LASFNet/data", exist_ok=True)
yaml_path = "/content/LASFNet/data/dronevehicle_new.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"File konfigurasi YAML sukses diperbarui di: {yaml_path}")

File konfigurasi YAML sukses diperbarui di: /content/LASFNet/data/dronevehicle_new.yaml


In [12]:
import os

base_dir = "/content/LASFNet/dataset/dronevehicle_new"
paths_to_check = [
    "data_rgb/train/images/14035.jpg",
    "data_ir/train/images/14035.jpg",
    "data_rgb/train/labels/14035.txt",
    "data_ir/train/labels/14035.txt"
]

print("=== VERIFIKASI FISIK FILE GHOST (04895) ===")
for p in paths_to_check:
    full_path = os.path.join(base_dir, p)
    if os.path.exists(full_path):
        print(f"File '{p}': ADA DI DISK! (Akan langsung dihapus sekarang...)")
        try:
            os.remove(full_path)
            print("  -> Berhasil dihapus!")
        except Exception as e:
            print(f"  -> Gagal menghapus: {e}")
    else:
        print(f"File '{p}': TIDAK ADA DI DISK (Bersih)")

print("\nJika semua status di atas sudah 'TIDAK ADA' namun training tetap error, silakan lakukan 'Runtime -> Restart Session' lalu jalankan ulang training.")

=== VERIFIKASI FISIK FILE GHOST (04895) ===
File 'data_rgb/train/images/14035.jpg': ADA DI DISK! (Akan langsung dihapus sekarang...)
  -> Berhasil dihapus!
File 'data_ir/train/images/14035.jpg': ADA DI DISK! (Akan langsung dihapus sekarang...)
  -> Berhasil dihapus!
File 'data_rgb/train/labels/14035.txt': ADA DI DISK! (Akan langsung dihapus sekarang...)
  -> Berhasil dihapus!
File 'data_ir/train/labels/14035.txt': ADA DI DISK! (Akan langsung dihapus sekarang...)
  -> Berhasil dihapus!

Jika semua status di atas sudah 'TIDAK ADA' namun training tetap error, silakan lakukan 'Runtime -> Restart Session' lalu jalankan ulang training.


### **LANGKAH 7: Mulai Proses Training LASFNet**

In [ ]:
%cd /content/LASFNet

SAVE_DIR = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run"

!python train.py \
  --imgsz 640 \
  --batch-size 48 \
  --epochs 300 \
  --data data/dronevehicle_new.yaml \
  --weights '' \
  --cfg models/LASFNet.yaml \
  --device 0 \
  --project "{SAVE_DIR}" \
  --name full_train_300ep \
  --save-period 20 \
  --exist-ok

Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
     21/299      37.6G    0.03952    0.06969   0.005015        967        640:  38% 142/375 [01:51<03:09,  1.23it/s]/content/LASFNet/train.py:420: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
     21/299      37.6G    0.03951    0.06972   0.005013       1115        640:  38% 143/375 [01:52<03:01,  1.28it/s]/content/LASFNet/train.py:420: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
     21/299      37.6G    0.03953    0.06967   0.005029        880        640:  38% 144/375 [01:52<02:30,  1.54it/s]/content/LASFNet/train.py:420: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autoca

### **LANGKAH OPSIONAL: Melanjutkan Training (Resume) Jika Sesi Terputus / Ganti Runtime**
Jika runtime terputus dan ingin melanjutkan training dari epoch terakhir yang sudah tersimpan di Google Drive, jalankan kode di bawah ini setelah Anda menjalankan ulang Langkah 1-6.

### **Perbaikan Perintah Resume Training**
Berikut adalah perintah yang benar untuk melakukan resume training tanpa memicu bentrok pembacaan file/direktori.

In [ ]:
%cd /content/LASFNet

LAST_WEIGHTS = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run/full_train_300ep/weights/last.pt"

# Perintah resume yang benar:
# 1. Masukkan path last.pt langsung ke dalam parameter --resume
# 2. Hapus parameter --weights dan --cfg agar tidak bentrok
!python train.py \
  --imgsz 640 \
  --batch-size 24 \
  --epochs 300 \
  --data data/dronevehicle_new.yaml \
  --resume "{LAST_WEIGHTS}" \
  --device 0 \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run" \
  --name full_train_300ep \
  --save-period 20 \
  --exist-ok

Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
    293/299      37.6G    0.03183    0.05152   0.001496       1002        640:  39% 148/375 [01:53<02:13,  1.71it/s]/content/LASFNet/train.py:436: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    293/299      37.6G    0.03182    0.05147   0.001495        921        640:  40% 149/375 [01:55<03:57,  1.05s/it]/content/LASFNet/train.py:436: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    293/299      37.6G    0.03183     0.0515   0.001493       1103        640:  40% 150/375 [01:55<03:09,  1.19it/s]/content/LASFNet/train.py:436: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autoca

### **LANGKAH 8: Evaluasi Validasi / Test using val.py**

In [13]:
%cd /content/LASFNet

# Eksekusi val.py dengan weights terbaik yang disimpan di Google Drive
!python val.py \
  --data data/dronevehicle_new.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/LASFNet/DroneVehicle/LASFNet_DroneVehicle_Run/full_train_300ep/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task test \
  --verbose

/content/LASFNet
/content/LASFNet/utils/general.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
YOLOv5 🚀 4620bb8 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)

Fusing layers... 
Model summary: 290 layers, 7976811 parameters, 0 gradients, 26.6 GFLOPs
Params: 7.7254M
FLOPs: 13.30G
100% 755k/755k [00:00<00:00, 164MB/s]
test: Scanning '/content/LASFNet/dataset/dronevehicle_new/data_rgb/val/labels' images and labels... 1469 found, 0 missing, 72 empty, 0 corrupted: 100% 1469/1469 [00:00<00:00, 2652.98it/s]
test: Scanning '/content/LASFNet/dataset/dronevehicle_new/data_ir/val/labels' images and labels... 1469 found, 0 missing, 47 empty, 0 corrupted: 100% 1469/1469 [00:00<00:00, 2733.99it/s]
                 Class     Images  Instances   

**End of Code**